# Task 3 - Retrieval: Popularity, Random, and MF-BPR

This notebook is the executable Task 3 pipeline.

It keeps concerns separated:
- model code lives in `scripts/models.py` and `scripts/mf_bpr.py`
- evaluation logic lives in `scripts/evaluate_task3.py`
- plot generation lives in `scripts/plot_task3_results.py`
- this notebook orchestrates the workflow and records the results

Run this notebook on a GPU runtime if you want the fastest MF-BPR training. The code still works on CPU with a smaller `max_train_positives` cap.

## Colab setup

Use these cells first when running in Google Colab. They clone the repo at the task branch and add the repo root to `sys.path`.

In [ ]:
# In Colab, replace REPO_URL with your remote repository URL.
REPO_URL = "https://github.com/<your-org-or-user>/steam-recsys-pipeline.git"
BRANCH = "task3-MF-BPR-Two-Tower"
CLONE_DIR = "/content/steam-recsys-pipeline"

!git clone --branch $BRANCH $REPO_URL $CLONE_DIR

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path('/content/steam-recsys-pipeline')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(REPO_ROOT)
print(sys.path[0])

In [ ]:
from pathlib import Path
import json
import pandas as pd

from scripts.evaluate_task3 import build_ground_truth, build_history, evaluate
from scripts.mf_bpr import MFBPRConfig
from scripts.plot_task3_results import load_results, plot_bars, plot_latency, plot_tradeoff

DATA_DIR = Path('.')
OUTPUT_DIR = Path('outputs/task3')
PLOTS_DIR = OUTPUT_DIR / 'plots'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load and inspect the processed splits

The project uses time-based splits already prepared in parquet format.

In [ ]:
train_df = pd.read_parquet(DATA_DIR / 'train.parquet')
val_df = pd.read_parquet(DATA_DIR / 'validation.parquet')
test_df = pd.read_parquet(DATA_DIR / 'test.parquet')

display(train_df.head())
print('train', train_df.shape)
print('validation', val_df.shape)
print('test', test_df.shape)
print('train positives', int(train_df['is_positive'].sum()))
print('validation positives', int(val_df['is_positive'].sum()))
print('test positives', int(test_df['is_positive'].sum()))

## 2. Build histories and ground truth

The evaluation uses training history for validation and training + validation history for test, matching the project protocol.

In [ ]:
val_history = build_history(train_df)
test_history = build_history(train_df, val_df)
val_gt = build_ground_truth(val_df)
test_gt = build_ground_truth(test_df)

val_users = [u for u, pos in val_gt.items() if pos]
test_users = [u for u, pos in test_gt.items() if pos]
catalog = set(train_df['item_id'].astype(str).unique())

print('catalog size', len(catalog))
print('validation users', len(val_users))
print('test users', len(test_users))

## 3. Configure MF-BPR

The model is implemented from scratch in NumPy. The config below is the main place to tune factors, learning rate, regularization, epochs, and the training cap for local GPU/CPU experiments.

In [ ]:
config = MFBPRConfig(
    n_factors=64,
    learning_rate=0.05,
    reg=0.0025,
    epochs=5,
    batch_size=2048,
    seed=42,
    device='cuda',
)
max_train_positives = 200000
print(config)
print('max_train_positives', max_train_positives)

## 4. Fit and evaluate the full Task 3 comparison

This step trains MF-BPR and evaluates all three retrievers on validation and test. The returned structure is also saved to JSON for plot generation and analysis.

In [ ]:
results = evaluate(
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    k=20,
    ndcg_k=10,
    mf_config=config,
    max_train_positives=max_train_positives,
)

results

## 5. Turn results into a comparison table

This is the table you can later paste into `ANALYSIS.md` or the README.

In [ ]:
results_df = load_results(OUTPUT_DIR / 'task3_results.json')
display(results_df)

## 6. Generate plots

The notebook generates the same plots as the standalone script so the full pipeline remains reproducible from the notebook alone.

In [ ]:
plot_bars(results_df, 'recall@20', PLOTS_DIR / 'recall_at_20.png')
plot_bars(results_df, 'ndcg@10', PLOTS_DIR / 'ndcg_at_10.png')
plot_bars(results_df, 'coverage@20', PLOTS_DIR / 'coverage_at_20.png')
plot_latency(results_df, PLOTS_DIR / 'latency.png')
plot_tradeoff(results_df, PLOTS_DIR / 'coverage_vs_recall.png')

print('plots saved to', PLOTS_DIR)

## 7. Save a compact summary

This creates a notebook-friendly artifact you can cite in the report.

In [ ]:
summary = {
    'config': config.__dict__,
    'max_train_positives': max_train_positives,
    'results': results,
}
with open(OUTPUT_DIR / 'task3_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, default=str)

print('saved', OUTPUT_DIR / 'task3_summary.json')